In [1]:
from brainsmash.workbench.geo import cortex
import os
from brainsmash.workbench.geo import volume
import nibabel as nib
import numpy as np
from brainsmash.mapgen.eval import sampled_fit


In [2]:
diag = "psy"

output_dir = "/scratch2/kg98/trangc/VBM/data/nulltest/volume_" + diag +'_index'# directory to which output binaries are written

if not os.path.exists(output_dir):
    os.mkdir(output_dir)

In [3]:
coord_file = "/projects/kg98/trangc/VBM/code/nulltest/pythonProject/mnimaskedtemplate_" + diag + "_index.txt"
# Load atlas
s132_img = nib.load('/projects/kg98/trangc/VBM/data/derivatives/s6/mask_' + diag +'/mask.nii')
atlas = s132_img.get_fdata()
# get one hemisphere
#atlas[20:, :, :] = 0
# Load subcortex atlas
# sub_img = nib.load('/projects/kg98/trangc/atlases/Tian_subcortical/CAT12MNI/Tian_Subcortex_S1_3T_2009cAsym_CAT12MNI.nii.gz')
# subcortex = sub_img.get_fdata()

# Load cerebellum atlas
# cere_img = nib.load('/projects/kg98/trangc/atlases/Human_cerebellum/Buckner-whole_1mm_CAT12MNI.nii.gz')
# cere = cere_img.get_fdata()

# Set subcortex and cerebellum regions to 0 in the cortex map
# cortex = atlas.copy()
# cortex[subcortex > 0] = 0
# cortex[cere > 0] = 0

# Combine output with s132_img header and save the cortex mask
# cortex_img = nib.Nifti1Image(cortex, s132_img.affine, s132_img.header)
# nib.save(cortex_img, "cortex_mask.nii.gz")

coords = np.column_stack(np.where(atlas > 0))
#np.savetxt(coord_file, coords, fmt='%d')
#print(coords)

In [ ]:
filenames = volume(coord_file, output_dir)

loading voxels coordinates from /projects/kg98/trangc/VBM/code/nulltest/pythonProject/mnimaskedtemplate_psy_index.txt
file contains 433287 voxels
saving memory-mapped distance matrix files to /scratch2/kg98/trangc/VBM/data/nulltest/volume_psy_index


In [5]:
condition = 'BD'
item = 'Baltimore'
# Conditional assignment
if condition == 'AD':
    diaggroup = 'AD'
else:
    diaggroup = 'psy'

distance_files = {'D': '/scratch2/kg98/trangc/VBM/data/nulltest/volume_'+ diaggroup +'_index/distmat.npy','index': '/scratch2/kg98/trangc/VBM/data/nulltest/volume_'+ diaggroup +'_index/index.npy'}

# smoothing kernel
smoothkernel = '6'

# Specify the directory path (replace with your actual path)
directory_path = '/projects/kg98/trangc/VBM/data/derivatives/s' + smoothkernel + 'COMBAT/' + condition

full_path = os.path.join(directory_path, item)  # Get the full path
if os.path.isdir(full_path):  # Check if the item is a directory
    # Step 1: Load the NIfTI file
    nii_file = full_path + '/' + 'spmT_0001.nii'  # Replace with your file path
    img = nib.load(nii_file)
    data = img.get_fdata()
      # Step 2: mask the data
    maskedData = data[atlas>0]

    # Step 3: Save the data to a text file
    brain_map = full_path + '/' + 'spmT_0001.txt'
   # np.savetxt(brain_map, maskedData, fmt='%.6f')  # Save with 6 decimal precision


In [6]:

print("Size of D:", np.load(distance_files['D']).shape)
print(maskedData.shape)


MemoryError: Unable to allocate 699. GiB for an array with shape (187737624369,) and data type float32

In [7]:
kwargs = {'ns': 500,
          'knn': 1500,
          'pv': 30,
          'kernel': 'gaussian',
          'n_jobs': 1
          }
          
(emp_var, u0, surr_var, surrogate_maps) = sampled_fit(x=brain_map, D=distance_files['D'],  
                                                      index=distance_files['index'],nsurr=2,return_data=True, **kwargs)



/scratch2/kg98/trangc/miniconda3/envs/trangcenv/lib/python3.8/site-packages/brainsmash/mapgen/sampled.py:306: RuntimeWarning: invalid value encountered in divide
  output = num / denom
/scratch2/kg98/trangc/miniconda3/envs/trangcenv/lib/python3.8/site-packages/brainsmash/mapgen/kernels.py:30: RuntimeWarning: invalid value encountered in divide
  return np.exp(-1.25 * np.square(d / d.max(axis=-1)[:, np.newaxis]))


ValueError: Input X contains NaN.
LinearRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [9]:
print(maskedData.shape)
print(surrogate_maps.shape)

(433287,)
(2, 433287)


In [7]:
# fill in the surrogate map values
null_map = np.zeros_like(atlas, dtype=float)
null_map[atlas>0] = surrogate_maps[1]

# Combine output with s132_img header and save
null_img = nib.Nifti1Image(null_map, s132_img.affine, s132_img.header)
nib.save(null_img, "null_map_guassian" + str(
    kwargs['ns']) + '_' + str(kwargs['knn']) + '_' + str(kwargs['pv']) +  "2.nii.gz")